# Вариант 1

In [8]:
import pandas as pd
import numpy as np

try:
    import micropip
    await micropip.install("openpyxl")
except (ImportError, AttributeError):
    pass

# Практическая работа 6
### Подготовка фич и интеграция источников

**Цель:** Научиться объединять несколько источников данных (заказы, товары, события) и строить на их основе признаки (фичи) для последующей аналитики и простых ML-моделей.

### Исходные данные

- Файлы заказов за три месяца: `data/orders_2024_01.csv`, `data/orders_2024_02.csv`, `data/orders_2024_03.csv`
- Файл товаров и категорий: `data/products.xlsx` (листы `products` и `categories`)
- Файлы логов событий пользователей: `data/events_records.json`, `data/events_lines.json`

### Подготовка: Загрузка исходных данных

In [9]:
# 1. Загрузка заказов за три месяца
order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]
df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in order_files],
    ignore_index=True
)
df_orders["revenue"] = df_orders["price"] * df_orders["quantity"]

# 2. Загрузка товаров и категорий из Excel

sheets = pd.read_excel("data/products.xlsx", sheet_name=["products", "categories"])
df_products = sheets["products"]
df_categories = sheets["categories"]


# 3. Загрузка событий из JSON (records и lines)
df_events_records = pd.read_json("data/events_records.json", orient="records")
df_events_lines = pd.read_json("data/events_lines.json", lines=True)
df_events = pd.concat([df_events_records, df_events_lines], ignore_index=True)
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print("Заказов загружено:", len(df_orders))
print("Товаров загружено:", len(df_products))
print("Категорий загружено:", len(df_categories))
print("Событий загружено:", len(df_events))

Заказов загружено: 15000
Товаров загружено: 30
Категорий загружено: 5
Событий загружено: 2005


### Задание 1. Объединить заказы с товарами и категориями

In [10]:
# Объединение товаров со справочником категорий по category_id
df_prod_full = df_products.merge(
    df_categories,
    on="category_id",
    how="left",
    suffixes=("", "_cat")
)

# Объединение заказов с полной информацией о товарах по product_id
df_orders_prod = df_orders.merge(
    df_prod_full,
    on="product_id",
    how="left"
)

print("Форма объединенной таблицы заказов:", df_orders_prod.shape)
print("Столбцы таблицы:", list(df_orders_prod.columns))
df_orders_prod[["order_id", "user_id", "order_datetime", "product_id", "product_name", "category_name", "price", "quantity", "revenue"]].head()

Форма объединенной таблицы заказов: (15000, 14)
Столбцы таблицы: ['order_id', 'user_id', 'order_datetime', 'product_id', 'price', 'quantity', 'status', 'revenue', 'product_name', 'category_id', 'category_name', 'weight_kg', 'category_name_cat', 'description']


,order_id,user_id,order_datetime,product_id,product_name,category_name,price,quantity,revenue
0,1,1044,2024-01-03 18:39:26,26,Product 26,Sport,14038.15,1,14038.15
1,2,1079,2024-01-03 17:45:43,16,Product 16,Sport,9172.60,1,9172.60
2,3,1041,2024-01-06 22:46:38,25,Product 25,Sport,9035.26,2,18070.52
3,4,1083,2024-01-08 21:03:51,9,Product 9,Toys,15234.33,3,45702.99
4,5,1068,2024-01-03 23:26:53,24,Product 24,Electronics,4134.38,2,8268.76


### Задание 2. Посчитать выручку по каждой категории и найти топ-5 категорий

In [11]:
# Группировка по категориям с расчетом выручки, количества заказов и проданных единиц
category_metrics = (
    df_orders_prod
    .groupby("category_name")
    .agg(
        revenue_sum=("revenue", "sum"),
        orders_count=("order_id", "nunique"),
        items_count=("quantity", "sum"),
    )
    .sort_values("revenue_sum", ascending=False)
)

print("Выручка по всем категориям:")
print(category_metrics)

# Выделение топ-5 категорий
top_5_categories = category_metrics.head(5)
print("\nТоп-5 категорий по выручке:")
(top_5_categories)

Выручка по всем категориям:
                revenue_sum  orders_count  items_count
category_name                                         
Toys           1.160671e+08          4544        11337
Sport          1.003407e+08          3900         9680
Electronics    7.742044e+07          3023         7637
Fashion        5.196834e+07          2002         5048
Home           3.874713e+07          1531         3863

Топ-5 категорий по выручке:


,revenue_sum,orders_count,items_count
category_name,,,
Toys,1.160671e+08,4544,11337
Sport,1.003407e+08,3900,9680
Electronics,7.742044e+07,3023,7637
Fashion,5.196834e+07,2002,5048
Home,3.874713e+07,1531,3863


### Подготовка: Формирование базового набора признаков пользователей (user_features)

In [12]:
# 1. Агрегация показателей заказов по каждому пользователю
user_metrics = (
    df_orders_prod
    .groupby("user_id")
    .agg(
        orders_count=("order_id", "nunique"),
        items_count=("quantity", "sum"),
        revenue_sum=("revenue", "sum"),
        revenue_mean=("revenue", "mean"),
        first_order=("order_datetime", "min"),
        last_order=("order_datetime", "max"),
        cancelled_share=("status", lambda s: (s == "cancelled").mean()),
    )
)

# 2. Агрегация активности пользователей по типам событий
events_by_user = (
    df_events
    .groupby("user_id")
    .agg(
        events_total=("event_id", "count"),
        events_page_view=("event_type", lambda s: (s == "page_view").sum()),
        events_click=("event_type", lambda s: (s == "click").sum()),
        events_add_to_cart=("event_type", lambda s: (s == "add_to_cart").sum()),
        events_purchase=("event_type", lambda s: (s == "purchase").sum()),
        first_event=("event_time", "min"),
        last_event=("event_time", "max"),
    )
)

# 3. Объединение метрик заказов и событий в единую витрину user_features
user_features = user_metrics.merge(events_by_user, on="user_id", how="outer")
print("Форма базового набора user_features:", user_features.shape)
user_features.head()

Форма базового набора user_features: (100, 14)


,orders_count,items_count,revenue_sum,revenue_mean,first_order,last_order,cancelled_share,events_total,events_page_view,events_click,events_add_to_cart,events_purchase,first_event,last_event
user_id,,,,,,,,,,,,,,
1001,177,454,4435767.09,25060.831017,2024-01-02 02:51:37,2024-03-31 21:32:19,0.146893,29,3,9,4,11,2024-02-01 20:18:15,2024-03-31 22:29:02
1002,123,281,2830004.52,23008.166829,2024-01-02 12:57:34,2024-03-31 19:45:16,0.130081,25,4,6,3,8,2024-02-05 07:27:49,2024-03-31 21:24:08
1003,156,389,4025266.18,25802.988333,2024-01-02 02:02:44,2024-03-31 19:36:26,0.121795,21,3,3,2,8,2024-02-09 07:00:19,2024-03-29 20:53:24
1004,147,380,3788030.56,25768.915374,2024-01-01 00:19:34,2024-03-31 20:01:40,0.115646,22,2,3,4,9,2024-02-03 21:50:28,2024-03-27 04:58:11
1005,141,346,3185653.37,22593.286312,2024-01-01 01:11:51,2024-03-31 18:16:10,0.163121,21,2,5,4,7,2024-02-02 13:55:51,2024-03-30 04:58:21


### Задание 3. Добавить в user_features признак «доля выручки из топ-категории» для каждого пользователя

In [13]:
# 1. Расчёт выручки по категориям для каждого пользователя
user_cat = (
    df_orders_prod
    .groupby(["user_id", "category_name"])
    .agg(revenue_sum=("revenue", "sum"))
    .reset_index()
)

# 2. Определение наиболее доходной (топ) категории каждого пользователя
user_top_cat = (
    user_cat
    .sort_values(["user_id", "revenue_sum"], ascending=[True, False])
    .drop_duplicates("user_id")
    .rename(columns={"category_name": "top_category", "revenue_sum": "top_category_revenue"})
)

# Добавляем персональную топ-категорию и долю выручки от неё
user_features = user_features.merge(user_top_cat, on="user_id", how="left")
user_features["top_category_share"] = (
    user_features["top_category_revenue"] / user_features["revenue_sum"]
).round(4)

# 3. Также рассчитываем долю выручки из топ-1 категории интернет-магазина (Toys)
global_top_1_cat = category_metrics.index[0]
global_top_cat_user = (
    user_cat[user_cat["category_name"] == global_top_1_cat]
    .rename(columns={"revenue_sum": f"revenue_{global_top_1_cat}"})
    [["user_id", f"revenue_{global_top_1_cat}"]]
)
user_features = user_features.merge(global_top_cat_user, on="user_id", how="left")
user_features[f"revenue_{global_top_1_cat}"] = user_features[f"revenue_{global_top_1_cat}"].fillna(0)
user_features["global_top_category_share"] = (
    user_features[f"revenue_{global_top_1_cat}"] / user_features["revenue_sum"]
).round(4)

print(f"Признаки добавлены! Глобальная категория №1: {global_top_1_cat}")
print(user_features[["orders_count", "revenue_sum", "top_category", "top_category_revenue", "top_category_share", "global_top_category_share"]].head(10))

Признаки добавлены! Глобальная категория №1: Toys
   orders_count  revenue_sum top_category  top_category_revenue  \
0           177   4435767.09         Toys            1238283.37   
1           123   2830004.52         Toys             902481.68   
2           156   4025266.18         Toys            1448523.72   
3           147   3788030.56        Sport            1111600.01   
4           141   3185653.37         Toys            1021461.33   
5           161   4160640.89         Toys            1513177.28   
6           157   4064170.75         Toys            1214503.49   
7           122   2886592.62         Toys            1064621.64   
8           147   3545781.73        Sport            1027725.41   
9           155   4240012.44         Toys            1203500.00   

   top_category_share  global_top_category_share  
0              0.2792                     0.2792  
1              0.3189                     0.3189  
2              0.3599                     0.3599  
3       

### Сохранение итогового набора признаков (user_features.csv)

In [14]:
# Сохранение датасета признаков в CSV
output_path = "data/user_features.csv"
user_features.to_csv(output_path, index=True)

print(f"Файл успешно сохранён: {output_path}")
print("Итоговая размерность датасета признаков:", user_features.shape)
print("Список сформированных признаков:")
for i, col in enumerate(user_features.columns, 1):
    print(f"  {i:>2}. {col}")

Файл успешно сохранён: data/user_features.csv
Итоговая размерность датасета признаков: (100, 20)
Список сформированных признаков:
   1. user_id
   2. orders_count
   3. items_count
   4. revenue_sum
   5. revenue_mean
   6. first_order
   7. last_order
   8. cancelled_share
   9. events_total
  10. events_page_view
  11. events_click
  12. events_add_to_cart
  13. events_purchase
  14. first_event
  15. last_event
  16. top_category
  17. top_category_revenue
  18. top_category_share
  19. revenue_Toys
  20. global_top_category_share
